## Cosmic Web - CppJIT - CUDA Kernel

[CppJIT](https://github.com/compiler-research/cppjit) compiles C++ and CUDA code inside the Python session. We first call CUB's histogram from C++, then launch our own kernel, passing CuPy's arrays and stream directly.

### Table of Contents

1. [Setup](#1-setup)
2. [CUB from C++](#2-cub-from-c)
3. [Kernel](#3-kernel)
4. [Launch](#4-launch)
5. [Trace and memory traffic](#5-trace-and-memory-traffic)
6. [Limitations](#6-limitations)

---

### 1. Setup

`import cppjit` starts a CUDA-enabled C++ interpreter ([clang-repl](https://clang.llvm.org/docs/ClangRepl.html)) in this process.

---

**Quick Docs**

- `cppjit.cppdef(src)`: compiles C++ or CUDA source into the session. Its functions and kernels become automatically available as Python proxies under the global namespace: `cppjit.gbl.<name>`.

In [ ]:
import importlib.metadata
import os
import time

import numpy as np
import cupy as cp

import cosmic_core as cw
import cppjit

assert cppjit.CUDA_ENABLED, 'this notebook needs the CUDA-enabled tutorial image'

N, K = cw.setup('cppjit_cub', f"cppjit {importlib.metadata.version('cppjit')}, "
                                  f"interpreter arguments --cuda {os.environ['CPPINTEROP_EXTRA_INTERPRETER_ARGS']}")

### 2. CUB from C++

Let's call `DeviceHistogram` from [CUB](https://github.com/NVIDIA/cccl/tree/main/cub) directly, in place of `cp.bincount`. `cub_histogram.cu` wraps it in a function that takes GPU pointers and the stream as integers.

---

**Quick Docs**

- `cppjit.gbl.cub_histogram(samples_ptr, hist_ptr, n, k_bins, stream)`: runs CUB's `HistogramEven` over `n` int32 samples into `k_bins` counters.
- `a.data.ptr`, `stream.ptr`: a CuPy array's GPU address and a stream's handle, as integers.

In [ ]:
t0 = time.perf_counter()
cppjit.cppdef(open('cub_histogram.cu').read())
cppdef_cub_s = time.perf_counter() - t0
print(f'cppdef of the CUB wrapper: {cppdef_cub_s * 1e3:.0f} ms')

SIGNS = (+1, -1, -1, -1, +1, +1, +1, -1)
CUB_HISTS = [cp.zeros(K, dtype=cp.int32) for _ in SIGNS]


def ecc_hist_cub(q):
    '''Signed histogram of births (host int64, length K): the CuPy stencil, CUB's histogram from C++ per cell array.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = cw.ecc_cells_cupy(q)
    stream = cp.cuda.get_current_stream()
    for a, h in zip((v, ex, ey, ez, fxy, fxz, fyz, c), CUB_HISTS):
        cppjit.gbl.cub_histogram(a.data.ptr, h.data.ptr, a.size, K, stream.ptr)
    # On the CPU, add the vertex and face counts and subtract the edge and cube counts.
    return sum(sign * cp.asnumpy(h).astype(np.int64) for sign, h in zip(SIGNS, CUB_HISTS))


def run_cub(n):
    return ecc_hist_cub(cw.field(n))


_ = cw.measure('cppjit_cub', run_cub, N, compile_s=cppdef_cub_s, note='incl. cppdef')
cw.compare('cppjit_cub', 'cupy')

- `HistogramEven` takes the bin count, so `cp.bincount`'s sign check, maximum and CPU waits are gone.
- `HistogramEven` counts one array per call, so the eight cell arrays take eight passes.
- The eight cell arrays still come from the CuPy stencil, with all its temporaries.

### 3. Kernel

How do we remove both costs? The numba kernel of the previous notebook computes each thread's births without writing them to memory and counts them in one pass. `ecc_histogram.cu` is the same algorithm in CUDA C++:

- **Compute births.** Each thread loads its $2\times2\times2$ window and takes the eight births with `min`.
- **Count in shared memory.** Each thread adds $\pm 1$ at each birth bin in the block's `__shared__` histogram.
- **Flush.** After `__syncthreads()`, the block adds its non-zero counters into the global histogram with `atomicAdd`.

Let's compile `ecc_histogram.cu` into the session.

---

**Quick Docs**

- `__global__ void kernel(...)`: a [CUDA C++](https://docs.nvidia.com/cuda/cuda-programming-guide/) kernel, launched on a grid of thread blocks. `blockIdx`, `blockDim` and `threadIdx` give each thread its index.
- `atomicAdd(int*, int)`, `atomicSub(int*, int)`: atomic add or subtract at an address in shared or global memory.

In [ ]:
t0 = time.perf_counter()
cppjit.cppdef(open('ecc_histogram.cu').read())
cppdef_s = time.perf_counter() - t0
print(f'cppdef of the kernel: {cppdef_s * 1e3:.0f} ms')

### 4. Launch

Let's launch the kernel with one thread per voxel, on CuPy's current stream. The shared histogram is static, so the dynamic shared memory is zero.

---

**Quick Docs**

- `cppjit.gbl.<kernel>[grid, block, shared_mem, stream](*args)`: launches the kernel. `grid` and `block` are ints or 3-tuples, `shared_mem` is the dynamic shared memory in bytes, and `stream` is CuPy's stream.

In [ ]:
BLOCK = 256


def ecc_hist_cppjit(q):
    '''Signed histogram of births (int32, length K) on the device: one launch on the current stream.'''
    n = q.shape[0]
    hist = cp.zeros(K, dtype=cp.int32)
    stream = cp.cuda.get_current_stream()
    grid = (n ** 3 + BLOCK - 1) // BLOCK
    cppjit.gbl.ecc_hist_kernel[grid, BLOCK, 0, stream](q, hist, n)
    return hist


def run_cppjit(n):
    hist = ecc_hist_cppjit(cw.field(n))
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist).astype(np.int64)

Now let's run a measurement. The first call includes three compiles: the `cppdef` above, CppJIT's kernel launcher, and the driver's compile of the PTX, cached after the first run.

In [ ]:
result = cw.measure('cppjit_cuda', run_cppjit, N, compile_s=cppdef_s, note='incl. cppdef')
cw.compare('cppjit_cuda', 'cppjit_cub')

### 5. Trace and memory traffic

Does one call launch one kernel, one memset and one copy back? Seven of each thread's eight loads should hit in cache, so we estimate one read of the field, four bytes per voxel.

In [ ]:
trace = cw.trace_call('cppjit_cuda', run_cppjit, N, [ecc_hist_cppjit], header=f'''
import cppjit
BLOCK = {BLOCK}
cppjit.cppdef(open('ecc_histogram.cu').read())
''', quiet=True)
cub = cw.trace_call('cppjit_cub', run_cub, N, [ecc_hist_cub], header="import cppjit\ncppjit.cppdef(open('cub_histogram.cu').read())",
                    state=[f'SIGNS = {SIGNS}', 'CUB_HISTS = [cp.zeros(K, dtype=cp.int32) for _ in SIGNS]'], quiet=True)
calls = {'CUB from C++': cub, 'the kernel': trace}
cw.call_table(calls)
cw.plot_timelines(calls, title=f'one call at N = {N}^3: CUB from C++ against the kernel')
cw.traffic_report('cppjit_cuda', result)
cw.save_sweep('cppjit_cuda', run_cppjit)

**Profiling the kernel**

Why is the measured time above this estimate? [Nsight Compute](https://docs.nvidia.com/nsight-compute/) replays one launch and reads the GPU's hardware counters: how busy the SMs and the memory system were, what the warps waited for, and which level of the memory hierarchy served each load.

---

**Quick Docs**

- `cw.ncu_report(stage, n, title, ...)`: profiles one launch with `ncu --set full` and prints the throughput, memory-hierarchy and stall metrics.

In [ ]:
cw.ncu_report('cppjit_cuda', N, f'one launch at N = {N}^3:')

What does the report say?

- SM throughput is higher than memory throughput: instruction issue limits the kernel.
- The top stall reason, `wait`, is a fixed-latency dependency on the previous instruction. Here it comes from the integer divisions that turn the thread index into $(i, j, k)$. The rows kernel below removes them.
- Each thread loads eight corners, so the kernel requests the field several times over. Most requests hit in L1 or L2, and the field is read from DRAM about once, as estimated.
- The per-voxel atomics stay in shared memory. The flush runs as reductions in L2.

What if the grid gave each thread its $i$ and $j$ directly? `ecc_hist_rows_kernel` in `ecc_histogram_rows.cu` runs the same body on a 3D grid of `(1, n, n)` thread blocks:

- **Grid.** One thread block per row: `blockIdx.z` gives $i$ and `blockIdx.y` gives $j$.
- **Threads.** Each thread walks several voxels along $k$. The block size sets how many, so we sweep it, then profile one.

In [ ]:
cppjit.cppdef(open('ecc_histogram_rows.cu').read())


def ecc_hist_rows(q, block):
    '''Signed histogram of births (int32, length K) on the device, through ecc_hist_rows_kernel.'''
    n = q.shape[0]
    hist = cp.zeros(K, dtype=cp.int32)
    stream = cp.cuda.get_current_stream()
    cppjit.gbl.ecc_hist_rows_kernel[(1, n, n), block, 0, stream](q, hist, n)
    return hist


def run_rows(n, block):
    hist = ecc_hist_rows(cw.field(n), block)
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist).astype(np.int64)

In [ ]:
print(f"{'block':>6} {'voxels/thread':>13} {'median ms':>9} {'GB/s field':>11}")
for block in (512, 256, 128, 64):
    r = cw.measure('rows', run_rows, N, block, record=False, quiet=True)
    print(f'{block:>6} {(N + block - 1) // block:>13} {r["median_s"] * 1e3:>9.3f} {cw.field(N).nbytes / r["median_s"] / 1e9:>11.0f}')

ROWS_BLOCK = 128
cw.ncu_report('rows', N, f'\nthe rows kernel, {(N + ROWS_BLOCK - 1) // ROWS_BLOCK} voxels per thread, one launch at N = {N}^3:',
              run=run_rows, functions=[ecc_hist_rows], args=(ROWS_BLOCK,), header='''
import cppjit
cppjit.cppdef(open('ecc_histogram_rows.cu').read())
''')

- Memory throughput is now the higher of the two. The top stall, `long_scoreboard`, is a wait on a global-memory load: memory limits the kernel.
- With more voxels per thread the field-read rate climbs, then flattens below the copy bandwidth.

Takeaways:

- The shared-memory histogram turns $8N^3$ global atomics into at most $K$ per thread block.
- CuPy's arrays and stream passed into CUB and our hand-written kernel with no copies or binding code.

### 6. Limitations

A field larger than the GPU's memory is a future direction: slab decomposition with a halo plane.

Next: `95__cosmic_web__synthesis.ipynb` compares the tools using the results in `timings/`.